# 객체 데이터셋 전처리_Local

`C:/Dataset/Train2YOLO_Outdoor_Raw`의 실외 객체 polygon 라벨만 사용해 YOLO detection용 bbox 데이터셋을 만듭니다.

생성 결과:

```text
C:/Dataset/Train2YOLO_Outdoor_ObjectBox/
├─ images/train
├─ images/val
├─ labels/train
├─ labels/val
├─ visualizations
└─ data.yaml
```

Raw polygon은 외접 bbox로 변환합니다. 지면 라벨(`outdoor_surface__...`)은 제외합니다.


In [1]:
from pathlib import Path
from collections import Counter
import os
import shutil
import yaml

RAW_ROOT = Path("C:/Dataset/Train2YOLO_Outdoor_Raw")
RAW_DATA_YAML = RAW_ROOT / "raw_data.yaml"
OBJECT_ROOT = Path("C:/Dataset/Train2YOLO_Outdoor_ObjectBox")
OBJECT_DATA_YAML = OBJECT_ROOT / "data.yaml"

OVERWRITE_LABELS = True
VIS_SAMPLES_PER_SPLIT = 20

if not RAW_DATA_YAML.exists():
    raise FileNotFoundError(RAW_DATA_YAML)

print("RAW_ROOT:", RAW_ROOT)
print("OBJECT_ROOT:", OBJECT_ROOT)


RAW_ROOT: C:\Dataset\Train2YOLO_Outdoor_Raw
OBJECT_ROOT: C:\Dataset\Train2YOLO_Outdoor_ObjectBox


## 객체 라벨 통합 기준

라즈베리파이 보행 보조에서 필요한 객체/장애물 중심으로 10개 클래스로 통합합니다.

- `vehicle`: 차량 + 이륜/킥보드 계열
- `mobility_aid`: 휠체어 + 유모차/캐리어
- `traffic_sign`: 교통 표지 + 제어함 + 소방시설 + 화분
- `temporary_obstacle`: 바리케이드, 입간판, 의자, 테이블, 키오스크


In [2]:
raw_data = yaml.safe_load(RAW_DATA_YAML.read_text(encoding="utf-8"))
raw_names = raw_data["names"]
if isinstance(raw_names, dict):
    raw_names = [raw_names[i] for i in sorted(raw_names)]

CLASS_NAMES = [
    "person",
    "vehicle",
    "mobility_aid",
    "animal",
    "vertical_obstacle",
    "temporary_obstacle",
    "bench",
    "traffic_light",
    "traffic_sign",
    "bus_taxi_stop",
]

RAW_TO_OBJECT = {
    "outdoor_polygon__person": "person",

    # 차량 + 이륜/킥보드 계열을 vehicle로 통합합니다.
    "outdoor_polygon__car": "vehicle",
    "outdoor_polygon__truck": "vehicle",
    "outdoor_polygon__bus": "vehicle",
    "outdoor_polygon__bicycle": "vehicle",
    "outdoor_polygon__motorcycle": "vehicle",
    "outdoor_polygon__scooter": "vehicle",

    # 휠체어와 이동 보조/운반 물체를 mobility_aid로 통합합니다.
    "outdoor_polygon__wheelchair": "mobility_aid",
    "outdoor_polygon__stroller": "mobility_aid",
    "outdoor_polygon__carrier": "mobility_aid",

    "outdoor_polygon__dog": "animal",
    "outdoor_polygon__cat": "animal",

    "outdoor_polygon__pole": "vertical_obstacle",
    "outdoor_polygon__bollard": "vertical_obstacle",
    "outdoor_polygon__tree_trunk": "vertical_obstacle",
    "outdoor_polygon__parking_meter": "vertical_obstacle",

    "outdoor_polygon__barricade": "temporary_obstacle",
    "outdoor_polygon__movable_signage": "temporary_obstacle",
    "outdoor_polygon__chair": "temporary_obstacle",
    "outdoor_polygon__table": "temporary_obstacle",
    "outdoor_polygon__kiosk": "temporary_obstacle",

    "outdoor_polygon__bench": "bench",
    "outdoor_polygon__traffic_light": "traffic_light",

    # 표지/제어함/소방시설/화분은 고정 시설물 성격으로 traffic_sign에 통합합니다.
    "outdoor_polygon__traffic_sign": "traffic_sign",
    "outdoor_polygon__traffic_light_controller": "traffic_sign",
    "outdoor_polygon__power_controller": "traffic_sign",
    "outdoor_polygon__fire_hydrant": "traffic_sign",
    "outdoor_polygon__potted_plant": "traffic_sign",

    "outdoor_polygon__stop": "bus_taxi_stop",
}

CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
RAW_ID_TO_NEW_ID = {}
for raw_name, merged_name in RAW_TO_OBJECT.items():
    if raw_name not in raw_names:
        print("Raw 데이터에 없는 라벨:", raw_name)
    else:
        RAW_ID_TO_NEW_ID[raw_names.index(raw_name)] = CLASS_TO_ID[merged_name]

print("최종 객체 클래스")
for i, name in enumerate(CLASS_NAMES):
    print(f"{i:02d}: {name}")

print("\nRaw -> Object 매핑")
for raw_id, new_id in sorted(RAW_ID_TO_NEW_ID.items()):
    print(f"{raw_id:02d} {raw_names[raw_id]} -> {new_id:02d} {CLASS_NAMES[new_id]}")


최종 객체 클래스
00: person
01: vehicle
02: mobility_aid
03: animal
04: vertical_obstacle
05: temporary_obstacle
06: bench
07: traffic_light
08: traffic_sign
09: bus_taxi_stop

Raw -> Object 매핑
23 outdoor_polygon__traffic_light -> 07 traffic_light
24 outdoor_polygon__pole -> 04 vertical_obstacle
25 outdoor_polygon__bollard -> 04 vertical_obstacle
26 outdoor_polygon__truck -> 01 vehicle
27 outdoor_polygon__person -> 00 person
28 outdoor_polygon__tree_trunk -> 04 vertical_obstacle
29 outdoor_polygon__traffic_light_controller -> 08 traffic_sign
30 outdoor_polygon__car -> 01 vehicle
31 outdoor_polygon__power_controller -> 08 traffic_sign
32 outdoor_polygon__traffic_sign -> 08 traffic_sign
33 outdoor_polygon__bicycle -> 01 vehicle
34 outdoor_polygon__bus -> 01 vehicle
35 outdoor_polygon__fire_hydrant -> 08 traffic_sign
36 outdoor_polygon__stop -> 09 bus_taxi_stop
37 outdoor_polygon__barricade -> 05 temporary_obstacle
38 outdoor_polygon__stroller -> 02 mobility_aid
39 outdoor_polygon__carrier -> 02

## 데이터셋 생성

YOLO detection 라벨 형식으로 저장합니다.

```text
class_id x_center y_center width height
```


In [3]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def ensure_dir(path):
    path.mkdir(parents=True, exist_ok=True)


def link_or_copy(src, dst):
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        return
    if dst.exists():
        dst.unlink()
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def parse_label_line(line):
    parts = line.strip().split()
    if len(parts) < 5:
        return None, []
    cls_id = int(float(parts[0]))
    values = [float(x) for x in parts[1:]]
    return cls_id, values


def clamp01(v):
    return max(0.0, min(1.0, v))


def polygon_to_bbox(values):
    if len(values) == 4:
        x, y, w, h = values
        if w <= 0 or h <= 0:
            return None
        return [clamp01(x), clamp01(y), clamp01(w), clamp01(h)]
    if len(values) < 6 or len(values) % 2 != 0:
        return None
    xs = [clamp01(v) for v in values[0::2]]
    ys = [clamp01(v) for v in values[1::2]]
    x1, x2 = min(xs), max(xs)
    y1, y2 = min(ys), max(ys)
    bw = x2 - x1
    bh = y2 - y1
    if bw <= 0 or bh <= 0:
        return None
    return [clamp01((x1 + x2) / 2), clamp01((y1 + y2) / 2), clamp01(bw), clamp01(bh)]


def format_box_line(cls_id, bbox):
    return " ".join([str(cls_id)] + [f"{v:.6f}" for v in bbox])


def convert_split(split):
    src_img_dir = RAW_ROOT / "images" / split
    src_lbl_dir = RAW_ROOT / "labels" / split
    dst_img_dir = OBJECT_ROOT / "images" / split
    dst_lbl_dir = OBJECT_ROOT / "labels" / split
    ensure_dir(dst_img_dir)
    ensure_dir(dst_lbl_dir)

    images = sorted(p for p in src_img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    total = len(images)
    stats = Counter()
    label_counts = Counter()

    for idx, img_path in enumerate(images, 1):
        if idx % 1000 == 0 or idx == total:
            print(f"{split}: {idx}/{total} ({idx / max(total, 1) * 100:.1f}%)")

        src_label = src_lbl_dir / f"{img_path.stem}.txt"
        if not src_label.exists():
            stats["missing_label"] += 1
            continue

        out_lines = []
        for line in src_label.read_text(encoding="utf-8").splitlines():
            raw_id, values = parse_label_line(line)
            if raw_id is None or raw_id not in RAW_ID_TO_NEW_ID:
                continue
            bbox = polygon_to_bbox(values)
            if bbox is None:
                stats["invalid_bbox"] += 1
                continue
            new_id = RAW_ID_TO_NEW_ID[raw_id]
            out_lines.append(format_box_line(new_id, bbox))
            label_counts[new_id] += 1

        if out_lines:
            link_or_copy(img_path, dst_img_dir / img_path.name)
            dst_label = dst_lbl_dir / f"{img_path.stem}.txt"
            if OVERWRITE_LABELS or not dst_label.exists():
                dst_label.write_text("\n".join(out_lines) + "\n", encoding="utf-8")
            stats["object_images"] += 1
        else:
            stats["empty_object"] += 1

    return stats, label_counts

all_stats = {}
total_counts = Counter()
for split in ["train", "val"]:
    stats, counts = convert_split(split)
    all_stats[split] = stats
    total_counts.update(counts)

print("\n변환 통계")
for split, stats in all_stats.items():
    print(split, dict(stats))

print("\n클래스별 bbox 수")
for i, name in enumerate(CLASS_NAMES):
    print(f"{i:02d} {name}: {total_counts[i]}")


train: 1000/125096 (0.8%)
train: 2000/125096 (1.6%)
train: 3000/125096 (2.4%)
train: 4000/125096 (3.2%)
train: 5000/125096 (4.0%)
train: 6000/125096 (4.8%)
train: 7000/125096 (5.6%)
train: 8000/125096 (6.4%)
train: 9000/125096 (7.2%)
train: 10000/125096 (8.0%)
train: 11000/125096 (8.8%)
train: 12000/125096 (9.6%)
train: 13000/125096 (10.4%)
train: 14000/125096 (11.2%)
train: 15000/125096 (12.0%)
train: 16000/125096 (12.8%)
train: 17000/125096 (13.6%)
train: 18000/125096 (14.4%)
train: 19000/125096 (15.2%)
train: 20000/125096 (16.0%)
train: 21000/125096 (16.8%)
train: 22000/125096 (17.6%)
train: 23000/125096 (18.4%)
train: 24000/125096 (19.2%)
train: 25000/125096 (20.0%)
train: 26000/125096 (20.8%)
train: 27000/125096 (21.6%)
train: 28000/125096 (22.4%)
train: 29000/125096 (23.2%)
train: 30000/125096 (24.0%)
train: 31000/125096 (24.8%)
train: 32000/125096 (25.6%)
train: 33000/125096 (26.4%)
train: 34000/125096 (27.2%)
train: 35000/125096 (28.0%)
train: 36000/125096 (28.8%)
train: 37000/

## data.yaml 생성

In [4]:
data_yaml = {
    "path": str(OBJECT_ROOT),
    "train": "images/train",
    "val": "images/val",
    "names": {i: name for i, name in enumerate(CLASS_NAMES)},
}
OBJECT_DATA_YAML.write_text(yaml.safe_dump(data_yaml, allow_unicode=True, sort_keys=False), encoding="utf-8")

print(OBJECT_DATA_YAML.read_text(encoding="utf-8"))
for rel in ["images/train", "images/val", "labels/train", "labels/val"]:
    folder = OBJECT_ROOT / rel
    pattern = "*" if rel.startswith("images") else "*.txt"
    print(rel, len(list(folder.glob(pattern))) if folder.exists() else "MISSING")


path: C:\Dataset\Train2YOLO_Outdoor_ObjectBox
train: images/train
val: images/val
names:
  0: person
  1: vehicle
  2: mobility_aid
  3: animal
  4: vertical_obstacle
  5: temporary_obstacle
  6: bench
  7: traffic_light
  8: traffic_sign
  9: bus_taxi_stop

images/train 83356
images/val 9367
labels/train 83356
labels/val 9367


## 샘플 시각화

In [5]:
if VIS_SAMPLES_PER_SPLIT > 0:
    from PIL import Image, ImageDraw

    vis_dir = OBJECT_ROOT / "visualizations"
    ensure_dir(vis_dir)
    colors = ["red", "lime", "cyan", "yellow", "magenta", "orange", "white", "deepskyblue", "pink", "green"]

    for split in ["train", "val"]:
        img_dir = OBJECT_ROOT / "images" / split
        lbl_dir = OBJECT_ROOT / "labels" / split
        samples = sorted(p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)[:VIS_SAMPLES_PER_SPLIT]
        for img_path in samples:
            label_path = lbl_dir / f"{img_path.stem}.txt"
            if not label_path.exists():
                continue
            img = Image.open(img_path).convert("RGB")
            draw = ImageDraw.Draw(img)
            w, h = img.size
            for line in label_path.read_text(encoding="utf-8").splitlines():
                cls_id, values = parse_label_line(line)
                if cls_id is None or len(values) < 4:
                    continue
                x, y, bw, bh = values[:4]
                x1 = (x - bw / 2) * w
                y1 = (y - bh / 2) * h
                x2 = (x + bw / 2) * w
                y2 = (y + bh / 2) * h
                color = colors[cls_id % len(colors)]
                draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
                draw.text((x1, max(0, y1 - 12)), CLASS_NAMES[cls_id], fill=color)
            img.save(vis_dir / f"{split}_{img_path.name}")

    print("시각화 저장:", vis_dir)


시각화 저장: C:\Dataset\Train2YOLO_Outdoor_ObjectBox\visualizations


## Colab 업로드용 압축

학습은 Colab에서 진행하므로 생성된 데이터셋을 zip으로 압축합니다.


In [6]:
MAKE_ZIP = True
ZIP_PATH = OBJECT_ROOT.with_suffix(".zip")

if MAKE_ZIP:
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    shutil.make_archive(str(OBJECT_ROOT), "zip", root_dir=OBJECT_ROOT)
    print("zip 생성:", ZIP_PATH)
    print("크기 GB:", round(ZIP_PATH.stat().st_size / (1024 ** 3), 3))


zip 생성: C:\Dataset\Train2YOLO_Outdoor_ObjectBox.zip
크기 GB: 3.078
